# 2.7 tf.data 效能優化

這份 Notebook 示範 `cache()`、`prefetch()` 與 `AUTOTUNE` 的基本使用方式。

## 1. 載入套件與建立資料

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)
X, y = make_classification(n_samples=3000, n_features=20, n_informative=10, random_state=42)
X = X.astype('float32')
y = y.astype('float32')
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## 2. 建立前處理函式

optimized pipeline 會先做穩定前處理並 cache，再依訓練需求 shuffle、batch 與 prefetch。這樣可以重用前處理結果，同時避免 cache 固定住第一次 shuffle 的順序。


In [ ]:
@tf.autograph.experimental.do_not_convert
def preprocess(features, label):
    features = tf.cast(features, tf.float32)
    features = tf.math.tanh(features)
    return features, label

def make_basic_dataset(x, y, training=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=42)
    return ds.map(preprocess).batch(64)

def make_optimized_dataset(x, y, training=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(x), seed=42)
    return ds.batch(64).prefetch(tf.data.AUTOTUNE)


## 3. 比較資料管線迭代時間

In [ ]:
def iterate_time(ds, n_rounds=2):
    start = time.perf_counter()
    rows = 0
    for _ in range(n_rounds):
        for batch_x, batch_y in ds:
            rows += int(batch_x.shape[0])
    return time.perf_counter() - start, rows

basic_ds = make_basic_dataset(x_train, y_train)
optimized_ds = make_optimized_dataset(x_train, y_train)

basic_time, basic_rows = iterate_time(basic_ds)
optimized_time, optimized_rows = iterate_time(optimized_ds)

pd.DataFrame([
    {'pipeline': 'basic', 'seconds': basic_time, 'rows_seen': basic_rows},
    {'pipeline': 'optimized', 'seconds': optimized_time, 'rows_seen': optimized_rows},
])

## 4. 使用 optimized dataset 訓練模型

In [ ]:
train_ds = make_optimized_dataset(x_train, y_train)
test_ds = make_optimized_dataset(x_test, y_test, training=False)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, epochs=10, verbose=0)

pd.DataFrame([
    model.evaluate(train_ds, verbose=0, return_dict=True),
    model.evaluate(test_ds, verbose=0, return_dict=True),
], index=['train', 'test'])

## 5. 小結

`cache()`、`prefetch()` 與 `AUTOTUNE` 是最常用的 tf.data 效能起手式。實務上仍需依資料大小、記憶體與前處理成本調整。